# A6. Agreement across steps

Are samples at different x from one start consistent with each other?

See `docs/plans/experiments-plan.md`, entry A6. Results: `artifacts/results/dla_agreement/`.

## Run

From the repo root. `uv run erosion-sim list` shows what is already trained and evaluated.

**1. Data.** Skip any that already exist.

Local:

```sh
uv run erosion-sim dla-generate dla_master
```

Cluster:

```sh
sbatch slurm/dla_generate.sbatch dla_master
```

**2. Train**, in this order. Each resumes if stopped; a finished model is skipped.

Local:

```sh
uv run erosion-sim train dla_base
```

Cluster:

```sh
sbatch slurm/train.sbatch dla_base
```

**3. Evaluate**, once training has finished. It refuses to replace existing results.

Local:

```sh
uv run erosion-sim evaluate dla_agreement dla_base
```

Cluster:

```sh
sbatch slurm/evaluate.sbatch dla_agreement dla_base
```

Then run the cells below.

How to read the tables and plots: [`docs/guides/dla.md`](../../docs/guides/dla.md).

In [ ]:
import json

from erosion_sim.dla import plots
from erosion_sim.evaluation.results import load
from erosion_sim.paths import model_dir
from erosion_sim.training.train import plot_history

EVALUATION = "dla_agreement"  # configs/evaluations/dla_agreement.yaml
PREDICTORS = ["dla_base"]

results = load(EVALUATION, *PREDICTORS)
{p: {k: m[k] for k in ("time", "code", "slurm_job")} for p, m in results.manifest["predictors"].items()}

## Containment

Fraction of the smaller sample's cells missing from the larger one, for every pair of jumps from one start and one noise field. 0 is perfect agreement.

In [ ]:
plots.plot_containment(results)
{p: results.scalars[p]["containment"] for p in PREDICTORS}

## Chaining

The full count reached in 1, 2, or 4 equal jumps, against the simulator's continuations from the same shared starts.

In [ ]:
for metric in plots.METRIC_LABELS:
    plots.plot_chain(results, metric)

## Conclusion

_Written after reading the results above._